# Project 2

In [89]:
import math
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats as stats

## Primary task

Notes:
Hospital has created two temporary wards: Ward A (regular care) and Ward B (intensive care)
To accomodate the temporary wards, staff and beds are moved from a pre-existing ward, Ward C (inpatient care).

New patients arrive through Poisson process (independent of receiving ward) and Length-Of-Stay (LOS) is often log-normal.

Assumption:
Intensity of epidemic behaves as second-order ploynomial starting at t = 0 and ending in t = 365

Pseudocode

New patient coming in:
if intensive -> Ward B
    if Ward B occupied -> Ward A

if regular -> Ward A
    if Ward A occupied -> relocate to other hospital

if inpatient -> Ward C
    if Ward C occupied -> relocate to other hospital

### Defining parameter functions

In [90]:
def lambd_1(t):
    return -(1/3650) * t**2 + (1/10) * t

def lambd_2(t):
    return 1/5 * lambd_1(t)

def LOS(x):
    mu = np.log(x*np.sqrt(2))
    sigma = np.sqrt(np.log(2))

    return np.random.lognormal(mu,sigma)

### Manual arrival calculations

In [91]:
t_total = 365

lambd_3 = 6

a = -1/3650
b = 1/10
t_max = -b / (2*a)

lambd_max_1 = a * t_max**2 + b * t_max
lambd_max_2 = 1/5 * lambd_max_1
lambd_max_3 = lambd_3

arrivals_A = []
arrivals_B = []
arrivals_C = []

t = 0
while t < t_total:
    t += np.random.exponential(1/lambd_max_1)

    if t >= t_total:
        break

    if np.random.rand() < lambd_1(t)/lambd_max_1:
        arrivals_A.append(t)    

t = 0
while t < t_total:
    t += np.random.exponential(1/lambd_max_2)

    if t >= t_total:
        break

    if np.random.rand() < lambd_2(t)/lambd_max_2:
        arrivals_B.append(t)

t = 0
while t < t_total:
    t += np.random.exponential(1/lambd_3)

    if t >= t_total:
        break

    arrivals_C.append(t)


arrivalList = ([(t, "regular") for t in arrivals_A] + [(t, "intensive") for t in arrivals_B] + [(t, "inpatient") for t in arrivals_C])

arrivalList.sort()

In [92]:
arrived_regular = sum(1 for _,p in arrivalList if p=="regular")
arrived_intensive = sum(1 for _,p in arrivalList if p=="intensive")
arrived_inpatient = sum(1 for _,p in arrivalList if p=="inpatient")

print(arrived_regular)
print(arrived_intensive)
print(arrived_inpatient)

arrived_total = len(arrivalList)

print(arrived_total)

2217
454
2137
4808


### Making arrival funtions

In [93]:
def lambd_3(t):
    return 6

def lambd_max_1():
    a = -1/3650
    b = 1/10
    t_max = -b / (2*a)

    return a * t_max**2 + b * t_max

def lambd_max_2():
    return 1/5 * lambd_max_1()

def lambd_max_3():
    return lambd_3(t)

In [94]:
def arrivals(lambd,lambd_max,t_total):
    arrivals = []
    
    lambda_max = lambd_max()

    t = 0
    while t < t_total:
        t += np.random.exponential(1/lambda_max)

        if t >= t_total:
            break

        if np.random.rand() < lambd(t)/lambda_max:
            arrivals.append(t)    

    return arrivals

In [95]:
t_total = 365

arrivals_A = arrivals(lambd_1,lambd_max_1,t_total)
arrivals_B = arrivals(lambd_2,lambd_max_2,t_total)
arrivals_C = arrivals(lambd_3,lambd_max_3,t_total)

arrivalList = ([(t, "regular") for t in arrivals_A] + [(t, "intensive") for t in arrivals_B] + [(t, "inpatient") for t in arrivals_C])

arrivalList.sort()

In [96]:
arrived_regular = sum(1 for _,p in arrivalList if p=="regular")
arrived_intensive = sum(1 for _,p in arrivalList if p=="intensive")
arrived_inpatient = sum(1 for _,p in arrivalList if p=="inpatient")

print(arrived_regular)
print(arrived_intensive)
print(arrived_inpatient)

arrived_total = len(arrivalList)

print(arrived_total)

2212
472
2154
4838


### Bed allocation function

In [97]:
def beds(bA,bB,bC,arrivalList):
    BA = [0]*bA
    BB = [0]*bB
    BC = [0]*bC

    relocated_regular = 0
    relocated_intensive = 0
    relocated_inpatient = 0
    
    overflow_intensive = 0
    
    for t_arrival,patient_type in arrivalList:
        if patient_type == "regular":

            t_departure = t_arrival + LOS(4)

            vacanciesA = [x for x in BA if x <= t_arrival]
            
            if vacanciesA:
                indx = BA.index(vacanciesA[0])
                BA[indx] = t_departure
                
            else:
                relocated_regular += 1
        
        if patient_type == "intensive":

            t_departure = t_arrival + LOS(6)

            vacanciesB = [x for x in BB if x <= t_arrival]
            
            if vacanciesB:
                indx = BB.index(vacanciesB[0])
                BB[indx] = t_departure
              
            else:
                vacanciesA = [x for x in BA if x <= t_arrival]
            
                if vacanciesA:
                    indx = BA.index(vacanciesA[0])
                    BA[indx] = t_departure
                    
                    overflow_intensive += 1
                    
                else:
                    relocated_intensive += 1
        
        if patient_type == "inpatient":

            t_departure = t_arrival + LOS(5)

            vacanciesC = [x for x in BC if x <= t_arrival]
            
            if vacanciesC:
                indx = BC.index(vacanciesC[0])
                BC[indx] = t_departure
                
            else:
                relocated_inpatient += 1
        
    return (relocated_regular,
            relocated_intensive,
            relocated_inpatient,
            overflow_intensive)  

In [98]:
b_tot = 75

bA = 30
bB = 10
bC = b_tot-bA-bB

results = beds(bA,bB,bC,arrivalList)

print(results)

(1216, 127, 906, 94)


### Integrate arrival function and bed allocation funcion

In [99]:
def hospital(beds_A,beds_B,beds_C,t_total = 365):
    
    #Generate arrivals
    arrivals_A = arrivals(lambd_1,lambd_max_1,t_total)
    arrivals_B = arrivals(lambd_2,lambd_max_2,t_total)
    arrivals_C = arrivals(lambd_3,lambd_max_3,t_total)

    arrivalList = ([(t, "regular") for t in arrivals_A] + [(t, "intensive") for t in arrivals_B] + [(t, "inpatient") for t in arrivals_C])

    arrivalList.sort()
    
    arrived_regular = sum(1 for _,p in arrivalList if p=="regular")
    arrived_intensive = sum(1 for _,p in arrivalList if p=="intensive")
    arrived_inpatient = sum(1 for _,p in arrivalList if p=="inpatient")

    #Perform bed allocation
    bed_status_A = [0]*beds_A
    bed_status_B = [0]*beds_B
    bed_status_C = [0]*beds_C

    relocated_regular = 0
    relocated_intensive = 0
    relocated_inpatient = 0
    
    overflow_intensive = 0

    t_occupied_A = 0
    t_occupied_B = 0
    t_occupied_C = 0
    
    for t_arrival,patient_type in arrivalList:
        if patient_type == "regular":

            los = LOS(4)

            t_departure = t_arrival + los

            vacanciesA = [x for x in bed_status_A if x <= t_arrival]
            
            if vacanciesA:
                indx = bed_status_A.index(vacanciesA[0])
                bed_status_A[indx] = t_departure

                t_occupied_A += los
                
            else:
                relocated_regular += 1
        
        if patient_type == "intensive":

            los = LOS(6)

            t_departure = t_arrival + los

            vacanciesB = [x for x in bed_status_B if x <= t_arrival]
            
            if vacanciesB:
                indx = bed_status_B.index(vacanciesB[0])
                bed_status_B[indx] = t_departure

                t_occupied_B += los
              
            else:
                vacanciesA = [x for x in bed_status_A if x <= t_arrival]
            
                if vacanciesA:
                    indx = bed_status_A.index(vacanciesA[0])
                    bed_status_A[indx] = t_departure
                    
                    t_occupied_A += los

                    overflow_intensive += 1
                    
                else:
                    relocated_intensive += 1
        
        if patient_type == "inpatient":

            los = LOS(5)

            t_departure = t_arrival + los

            vacanciesC = [x for x in bed_status_C if x <= t_arrival]
            
            if vacanciesC:
                indx = bed_status_C.index(vacanciesC[0])
                bed_status_C[indx] = t_departure

                t_occupied_C += los
                
            else:
                relocated_inpatient += 1

    #Calculate total utilization of wards (time beds were occupied by patients / total "bed time" available)
    utilization_A = t_occupied_A / (beds_A*t_total)
    utilization_B = t_occupied_B / (beds_B*t_total)
    utilization_C = t_occupied_C / (beds_C*t_total)

    return (arrived_regular,
            arrived_intensive,
            arrived_inpatient,
            relocated_regular,
            relocated_intensive,
            relocated_inpatient,
            overflow_intensive,
            utilization_A,
            utilization_B,
            utilization_C) 

In [100]:
beds_total = 75

beds_A = 30
beds_B = 10
beds_C = beds_total - beds_A - beds_B

results = hospital(beds_A,beds_B,beds_C)

print(results)


(2234, 475, 2212, 1164, 145, 932, 95, 0.8537088600432461, 0.7745109442022099, 0.9781860617147642)


## Primary performance measures

### Perform repetitions

In [ ]:
reps = 10

total_arrived_regular = []
total_arrived_intensive = []
total_arrived_inpatient = []

total_relocated_regular = []
total_relocated_intensive = []
total_relocated_inpatient = []

total_overflow_intensive = []

total_utilization_A = []
total_utilization_B = []
total_utilization_C = []

for _ in range(reps):
    (arrived_regular,
     arrived_intensive,
     arrived_inpatient,
     relocated_regular,
     relocated_intensive,
     relocated_inpatient,
     overflow_intensive,
     utilization_A,
     utilization_B,
     utilization_C) = hospital(beds_A,beds_B,beds_C)

    total_arrived_regular.append(arrived_regular)
    total_arrived_intensive.append(arrived_intensive)
    total_arrived_inpatient.append(arrived_inpatient)

    total_relocated_regular.append(relocated_regular)
    total_relocated_intensive.append(relocated_intensive)
    total_relocated_inpatient.append(relocated_inpatient)

    total_overflow_intensive.append(overflow_intensive)

    total_utilization_A.append(utilization_A)
    total_utilization_B.append(utilization_B)
    total_utilization_C.append(utilization_C)

### Calculations

In [102]:
#Probability of all beds occupied on arrival
#Method 1:
total_p_occupied_regular = [relocated / arrived for relocated,arrived in zip(total_relocated_regular,total_arrived_regular)]
p_occupied_regular = np.mean(total_p_occupied_regular)

total_p_occupied_intensive = [(overflow + relocated) / arrived for overflow,relocated,arrived in zip(total_overflow_intensive,total_relocated_intensive,total_arrived_intensive)]
p_occupied_intensive = np.mean(total_p_occupied_intensive)

total_p_occupied_inpatient = [relocated / arrived for relocated,arrived in zip(total_relocated_inpatient,total_arrived_inpatient)]
p_occupied_inpatient = np.mean(total_p_occupied_inpatient)

print("p_occupied_regular (method 1):", p_occupied_regular)
print("p_occupied_intensive (method 1):", p_occupied_intensive)
print("p_occupied_inpatient (method 1):", p_occupied_inpatient)

#Method 2:
p_occupied_regular = (np.sum(total_relocated_regular) / np.sum(total_arrived_regular))
p_occupied_intensive = ((np.sum(total_overflow_intensive) + np.sum(total_relocated_intensive)) / np.sum(total_arrived_intensive))
p_occupied_inpatient = (np.sum(total_relocated_inpatient) / np.sum(total_arrived_inpatient))

print("p_occupied_regular (method 2):", p_occupied_regular)
print("p_occupied_intensive (method 2):", p_occupied_intensive)
print("p_occupied_inpatient (method 2):", p_occupied_inpatient)


#Mean number of relocated patients
mean_relocated_regular = np.mean(total_relocated_regular)
mean_relocated_intensive = np.mean(total_relocated_intensive)
mean_relocated_inpatient = np.mean(total_relocated_inpatient)

print("mean_relocated_regular:", mean_relocated_regular)
print("mean_relocated_intensive:", mean_relocated_intensive)
print("mean_relocated_inpatient:", mean_relocated_inpatient)

#Method 1:
total_relocated = [regular + intensive + inpatient for regular,intensive,inpatient in zip(total_relocated_regular,total_relocated_intensive,total_relocated_inpatient)]
mean_relocated = np.mean(total_relocated)

print("mean_relocated (method 1)", mean_relocated)

#Method 2:
mean_relocated = ((np.sum(total_relocated_regular) + np.sum(total_relocated_intensive) + np.sum(total_relocated_inpatient)) / reps)

print("mean_relocated (method 2):", mean_relocated)


#Mean fraction of beds utilized
mean_utilization_A = np.mean(total_utilization_A)
mean_utilization_B = np.mean(total_utilization_B)
mean_utilization_C = np.mean(total_utilization_C)

print("mean_utilization_A:", mean_utilization_A)
print("mean_utilization_B:", mean_utilization_B)
print("mean_utilization_C:", mean_utilization_C)

p_occupied_regular (method 1): 0.5224139496907464
p_occupied_intensive (method 1): 0.4633777533736228
p_occupied_inpatient (method 1): 0.42628948083075224
p_occupied_regular (method 2): 0.5224491632175634
p_occupied_intensive (method 2): 0.46383543463835436
p_occupied_inpatient (method 2): 0.4264303935860058
mean_relocated_regular: 1161.3
mean_relocated_intensive: 129.5
mean_relocated_inpatient: 936.1
mean_relocated (method 1) 2226.9
mean_relocated (method 2): 2226.9
mean_utilization_A: 0.8629668853617811
mean_utilization_B: 0.8056609863165389
mean_utilization_C: 0.9813166802899034


## Sensitivity analysis

In [103]:
reps = 3

best_sum = float('inf')
best_beds = None

for beds_A in range(1,beds_total):
    for beds_B in range(1,beds_total - beds_A):
        beds_C = beds_total - beds_A - beds_B

        relocated = []

        for _ in range(reps):
            (_,_,_,
            relocated_regular,
            relocated_intensive,
            relocated_inpatient,
            _,_,_,_) = hospital(beds_A, beds_B, beds_C)

            relocated.append(relocated_regular + relocated_intensive + relocated_inpatient)

        mean_relocated = np.mean(relocated)

        if mean_relocated < best_sum:
            best_sum = mean_relocated
            best_beds = (beds_A,beds_B,beds_C)

print(best_beds,best_sum)

beds_A = best_beds[0]
beds_B = best_beds[1]
beds_C = best_beds[2]

(30, 1, 44) 2034.6666666666667


### Repeat performance measures with optimized bed distribution

In [104]:
reps = 10

total_arrived_regular = []
total_arrived_intensive = []
total_arrived_inpatient = []

total_relocated_regular = []
total_relocated_intensive = []
total_relocated_inpatient = []

total_overflow_intensive = []

total_utilization_A = []
total_utilization_B = []
total_utilization_C = []

for _ in range(reps):
    (arrived_regular,
     arrived_intensive,
     arrived_inpatient,
     relocated_regular,
     relocated_intensive,
     relocated_inpatient,
     overflow_intensive,
     utilization_A,
     utilization_B,
     utilization_C) = hospital(beds_A,beds_B,beds_C)

    total_arrived_regular.append(arrived_regular)
    total_arrived_intensive.append(arrived_intensive)
    total_arrived_inpatient.append(arrived_inpatient)

    total_relocated_regular.append(relocated_regular)
    total_relocated_intensive.append(relocated_intensive)
    total_relocated_inpatient.append(relocated_inpatient)

    total_overflow_intensive.append(overflow_intensive)

    total_utilization_A.append(utilization_A)
    total_utilization_B.append(utilization_B)
    total_utilization_C.append(utilization_C)

In [105]:
#Probability of all beds occupied on arrival
#Method 1:
total_p_occupied_regular = [relocated / arrived for relocated,arrived in zip(total_relocated_regular,total_arrived_regular)]
p_occupied_regular = np.mean(total_p_occupied_regular)

total_p_occupied_intensive = [(overflow + relocated) / arrived for overflow,relocated,arrived in zip(total_overflow_intensive,total_relocated_intensive,total_arrived_intensive)]
p_occupied_intensive = np.mean(total_p_occupied_intensive)

total_p_occupied_inpatient = [relocated / arrived for relocated,arrived in zip(total_relocated_inpatient,total_arrived_inpatient)]
p_occupied_inpatient = np.mean(total_p_occupied_inpatient)

print("p_occupied_regular (method 1):", p_occupied_regular)
print("p_occupied_intensive (method 1):", p_occupied_intensive)
print("p_occupied_inpatient (method 1):", p_occupied_inpatient)

#Method 2:
p_occupied_regular = (np.sum(total_relocated_regular) / np.sum(total_arrived_regular))
p_occupied_intensive = ((np.sum(total_overflow_intensive) + np.sum(total_relocated_intensive)) / np.sum(total_arrived_intensive))
p_occupied_inpatient = (np.sum(total_relocated_inpatient) / np.sum(total_arrived_inpatient))

print("p_occupied_regular (method 2):", p_occupied_regular)
print("p_occupied_intensive (method 2):", p_occupied_intensive)
print("p_occupied_inpatient (method 2):", p_occupied_inpatient)


#Mean number of relocated patients
mean_relocated_regular = np.mean(total_relocated_regular)
mean_relocated_intensive = np.mean(total_relocated_intensive)
mean_relocated_inpatient = np.mean(total_relocated_inpatient)

print("mean_relocated_regular:", mean_relocated_regular)
print("mean_relocated_intensive:", mean_relocated_intensive)
print("mean_relocated_inpatient:", mean_relocated_inpatient)

#Method 1:
total_relocated = [regular + intensive + inpatient for regular,intensive,inpatient in zip(total_relocated_regular,total_relocated_intensive,total_relocated_inpatient)]
mean_relocated = np.mean(total_relocated)

print("mean_relocated (method 1)", mean_relocated)

#Method 2:
mean_relocated = ((np.sum(total_relocated_regular) + np.sum(total_relocated_intensive) + np.sum(total_relocated_inpatient)) / reps)

print("mean_relocated (method 2):", mean_relocated)


#Mean fraction of beds utilized
mean_utilization_A = np.mean(total_utilization_A)
mean_utilization_B = np.mean(total_utilization_B)
mean_utilization_C = np.mean(total_utilization_C)

print("mean_utilization_A:", mean_utilization_A)
print("mean_utilization_B:", mean_utilization_B)
print("mean_utilization_C:", mean_utilization_C)

p_occupied_regular (method 1): 0.5697185796084338
p_occupied_intensive (method 1): 0.9375612788647189
p_occupied_inpatient (method 1): 0.2899262435065441
p_occupied_regular (method 2): 0.5697272032717631
p_occupied_intensive (method 2): 0.9373861566484517
p_occupied_inpatient (method 2): 0.29002266524816134
mean_relocated_regular: 1267.7
mean_relocated_intensive: 243.2
mean_relocated_inpatient: 627.0
mean_relocated (method 1) 2137.9
mean_relocated (method 2): 2137.9
mean_utilization_A: 0.8887395848262573
mean_utilization_B: 0.9364987115337213
mean_utilization_C: 0.9692816573181284


### Implement exponential LOS and repeat performance measures

In [106]:
def LOS_exp(mean):
    return np.random.exponential(mean)

def hospital_exp(beds_A,beds_B,beds_C,t_total = 365):
    
    #Generate arrivals
    arrivals_A = arrivals(lambd_1,lambd_max_1,t_total)
    arrivals_B = arrivals(lambd_2,lambd_max_2,t_total)
    arrivals_C = arrivals(lambd_3,lambd_max_3,t_total)

    arrivalList = ([(t, "regular") for t in arrivals_A] + [(t, "intensive") for t in arrivals_B] + [(t, "inpatient") for t in arrivals_C])

    arrivalList.sort()
    
    arrived_regular = sum(1 for _,p in arrivalList if p=="regular")
    arrived_intensive = sum(1 for _,p in arrivalList if p=="intensive")
    arrived_inpatient = sum(1 for _,p in arrivalList if p=="inpatient")

    #Perform bed allocation
    bed_status_A = [0]*beds_A
    bed_status_B = [0]*beds_B
    bed_status_C = [0]*beds_C

    relocated_regular = 0
    relocated_intensive = 0
    relocated_inpatient = 0
    
    overflow_intensive = 0

    t_occupied_A = 0
    t_occupied_B = 0
    t_occupied_C = 0
    
    for t_arrival,patient_type in arrivalList:
        if patient_type == "regular":

            los = LOS_exp(8)

            t_departure = t_arrival + los

            vacanciesA = [x for x in bed_status_A if x <= t_arrival]
            
            if vacanciesA:
                indx = bed_status_A.index(vacanciesA[0])
                bed_status_A[indx] = t_departure

                t_occupied_A += los
                
            else:
                relocated_regular += 1
        
        if patient_type == "intensive":

            los = LOS_exp(12)

            t_departure = t_arrival + los

            vacanciesB = [x for x in bed_status_B if x <= t_arrival]
            
            if vacanciesB:
                indx = bed_status_B.index(vacanciesB[0])
                bed_status_B[indx] = t_departure

                t_occupied_B += los
              
            else:
                vacanciesA = [x for x in bed_status_A if x <= t_arrival]
            
                if vacanciesA:
                    indx = bed_status_A.index(vacanciesA[0])
                    bed_status_A[indx] = t_departure
                    
                    t_occupied_A += los

                    overflow_intensive += 1
                    
                else:
                    relocated_intensive += 1
        
        if patient_type == "inpatient":

            los = LOS_exp(10)

            t_departure = t_arrival + los

            vacanciesC = [x for x in bed_status_C if x <= t_arrival]
            
            if vacanciesC:
                indx = bed_status_C.index(vacanciesC[0])
                bed_status_C[indx] = t_departure

                t_occupied_C += los
                
            else:
                relocated_inpatient += 1

    #Calculate total utilization of wards (time beds were occupied by patients / total "bed time" available)
    utilization_A = t_occupied_A / (beds_A*t_total)
    utilization_B = t_occupied_B / (beds_B*t_total)
    utilization_C = t_occupied_C / (beds_C*t_total)

    return (arrived_regular,
            arrived_intensive,
            arrived_inpatient,
            relocated_regular,
            relocated_intensive,
            relocated_inpatient,
            overflow_intensive,
            utilization_A,
            utilization_B,
            utilization_C) 

In [107]:
reps = 10

total_arrived_regular = []
total_arrived_intensive = []
total_arrived_inpatient = []

total_relocated_regular = []
total_relocated_intensive = []
total_relocated_inpatient = []

total_overflow_intensive = []

total_utilization_A = []
total_utilization_B = []
total_utilization_C = []

for _ in range(reps):
    (arrived_regular,
     arrived_intensive,
     arrived_inpatient,
     relocated_regular,
     relocated_intensive,
     relocated_inpatient,
     overflow_intensive,
     utilization_A,
     utilization_B,
     utilization_C) = hospital_exp(beds_A,beds_B,beds_C)

    total_arrived_regular.append(arrived_regular)
    total_arrived_intensive.append(arrived_intensive)
    total_arrived_inpatient.append(arrived_inpatient)

    total_relocated_regular.append(relocated_regular)
    total_relocated_intensive.append(relocated_intensive)
    total_relocated_inpatient.append(relocated_inpatient)

    total_overflow_intensive.append(overflow_intensive)

    total_utilization_A.append(utilization_A)
    total_utilization_B.append(utilization_B)
    total_utilization_C.append(utilization_C)

In [108]:
#Probability of all beds occupied on arrival
#Method 1:
total_p_occupied_regular = [relocated / arrived for relocated,arrived in zip(total_relocated_regular,total_arrived_regular)]
p_occupied_regular = np.mean(total_p_occupied_regular)

total_p_occupied_intensive = [(overflow + relocated) / arrived for overflow,relocated,arrived in zip(total_overflow_intensive,total_relocated_intensive,total_arrived_intensive)]
p_occupied_intensive = np.mean(total_p_occupied_intensive)

total_p_occupied_inpatient = [relocated / arrived for relocated,arrived in zip(total_relocated_inpatient,total_arrived_inpatient)]
p_occupied_inpatient = np.mean(total_p_occupied_inpatient)

print("p_occupied_regular (method 1):", p_occupied_regular)
print("p_occupied_intensive (method 1):", p_occupied_intensive)
print("p_occupied_inpatient (method 1):", p_occupied_inpatient)

#Method 2:
p_occupied_regular = (np.sum(total_relocated_regular) / np.sum(total_arrived_regular))
p_occupied_intensive = ((np.sum(total_overflow_intensive) + np.sum(total_relocated_intensive)) / np.sum(total_arrived_intensive))
p_occupied_inpatient = (np.sum(total_relocated_inpatient) / np.sum(total_arrived_inpatient))

print("p_occupied_regular (method 2):", p_occupied_regular)
print("p_occupied_intensive (method 2):", p_occupied_intensive)
print("p_occupied_inpatient (method 2):", p_occupied_inpatient)


#Mean number of relocated patients
mean_relocated_regular = np.mean(total_relocated_regular)
mean_relocated_intensive = np.mean(total_relocated_intensive)
mean_relocated_inpatient = np.mean(total_relocated_inpatient)

print("mean_relocated_regular:", mean_relocated_regular)
print("mean_relocated_intensive:", mean_relocated_intensive)
print("mean_relocated_inpatient:", mean_relocated_inpatient)

#Method 1:
total_relocated = [regular + intensive + inpatient for regular,intensive,inpatient in zip(total_relocated_regular,total_relocated_intensive,total_relocated_inpatient)]
mean_relocated = np.mean(total_relocated)

print("mean_relocated (method 1)", mean_relocated)

#Method 2:
mean_relocated = ((np.sum(total_relocated_regular) + np.sum(total_relocated_intensive) + np.sum(total_relocated_inpatient)) / reps)

print("mean_relocated (method 2):", mean_relocated)


#Mean fraction of beds utilized
mean_utilization_A = np.mean(total_utilization_A)
mean_utilization_B = np.mean(total_utilization_B)
mean_utilization_C = np.mean(total_utilization_C)

print("mean_utilization_A:", mean_utilization_A)
print("mean_utilization_B:", mean_utilization_B)
print("mean_utilization_C:", mean_utilization_C)

p_occupied_regular (method 1): 0.5627216632538004
p_occupied_intensive (method 1): 0.9399094512837559
p_occupied_inpatient (method 1): 0.28673574240726235
p_occupied_regular (method 2): 0.5629765856284202
p_occupied_intensive (method 2): 0.9399466192170819
p_occupied_inpatient (method 2): 0.2872712410880523
mean_relocated_regular: 1255.1
mean_relocated_intensive: 240.1
mean_relocated_inpatient: 632.6
mean_relocated (method 1) 2127.8
mean_relocated (method 2): 2127.8
mean_utilization_A: 0.8885920329081621
mean_utilization_B: 0.9052251647864857
mean_utilization_C: 0.9667008445823939


### Varying total amount of beds

In [113]:
beds_optimal_A = beds_A
beds_optimal_B = beds_B
beds_optimal_C = beds_C
beds_optimal_total = beds_total

In [114]:
#Vary beds_total to 80, 100, 70 and 60
beds_total = 80

#Keep the same proportion as for optimal distribution
beds_A = round(beds_optimal_A/beds_optimal_total * beds_total)
beds_B = round(beds_optimal_B/beds_optimal_total * beds_total)
beds_C = beds_total - beds_A - beds_B

In [115]:
reps = 10

total_arrived_regular = []
total_arrived_intensive = []
total_arrived_inpatient = []

total_relocated_regular = []
total_relocated_intensive = []
total_relocated_inpatient = []

total_overflow_intensive = []

total_utilization_A = []
total_utilization_B = []
total_utilization_C = []

for _ in range(reps):
    (arrived_regular,
     arrived_intensive,
     arrived_inpatient,
     relocated_regular,
     relocated_intensive,
     relocated_inpatient,
     overflow_intensive,
     utilization_A,
     utilization_B,
     utilization_C) = hospital(beds_A,beds_B,beds_C)

    total_arrived_regular.append(arrived_regular)
    total_arrived_intensive.append(arrived_intensive)
    total_arrived_inpatient.append(arrived_inpatient)

    total_relocated_regular.append(relocated_regular)
    total_relocated_intensive.append(relocated_intensive)
    total_relocated_inpatient.append(relocated_inpatient)

    total_overflow_intensive.append(overflow_intensive)

    total_utilization_A.append(utilization_A)
    total_utilization_B.append(utilization_B)
    total_utilization_C.append(utilization_C)

In [116]:
#Probability of all beds occupied on arrival
#Method 1:
total_p_occupied_regular = [relocated / arrived for relocated,arrived in zip(total_relocated_regular,total_arrived_regular)]
p_occupied_regular = np.mean(total_p_occupied_regular)

total_p_occupied_intensive = [(overflow + relocated) / arrived for overflow,relocated,arrived in zip(total_overflow_intensive,total_relocated_intensive,total_arrived_intensive)]
p_occupied_intensive = np.mean(total_p_occupied_intensive)

total_p_occupied_inpatient = [relocated / arrived for relocated,arrived in zip(total_relocated_inpatient,total_arrived_inpatient)]
p_occupied_inpatient = np.mean(total_p_occupied_inpatient)

print("p_occupied_regular (method 1):", p_occupied_regular)
print("p_occupied_intensive (method 1):", p_occupied_intensive)
print("p_occupied_inpatient (method 1):", p_occupied_inpatient)

#Method 2:
p_occupied_regular = (np.sum(total_relocated_regular) / np.sum(total_arrived_regular))
p_occupied_intensive = ((np.sum(total_overflow_intensive) + np.sum(total_relocated_intensive)) / np.sum(total_arrived_intensive))
p_occupied_inpatient = (np.sum(total_relocated_inpatient) / np.sum(total_arrived_inpatient))

print("p_occupied_regular (method 2):", p_occupied_regular)
print("p_occupied_intensive (method 2):", p_occupied_intensive)
print("p_occupied_inpatient (method 2):", p_occupied_inpatient)


#Mean number of relocated patients
mean_relocated_regular = np.mean(total_relocated_regular)
mean_relocated_intensive = np.mean(total_relocated_intensive)
mean_relocated_inpatient = np.mean(total_relocated_inpatient)

print("mean_relocated_regular:", mean_relocated_regular)
print("mean_relocated_intensive:", mean_relocated_intensive)
print("mean_relocated_inpatient:", mean_relocated_inpatient)

#Method 1:
total_relocated = [regular + intensive + inpatient for regular,intensive,inpatient in zip(total_relocated_regular,total_relocated_intensive,total_relocated_inpatient)]
mean_relocated = np.mean(total_relocated)

print("mean_relocated (method 1)", mean_relocated)

#Method 2:
mean_relocated = ((np.sum(total_relocated_regular) + np.sum(total_relocated_intensive) + np.sum(total_relocated_inpatient)) / reps)

print("mean_relocated (method 2):", mean_relocated)


#Mean fraction of beds utilized
mean_utilization_A = np.mean(total_utilization_A)
mean_utilization_B = np.mean(total_utilization_B)
mean_utilization_C = np.mean(total_utilization_C)

print("mean_utilization_A:", mean_utilization_A)
print("mean_utilization_B:", mean_utilization_B)
print("mean_utilization_C:", mean_utilization_C)

p_occupied_regular (method 1): 0.4385754876173361
p_occupied_intensive (method 1): 0.7528453798997448
p_occupied_inpatient (method 1): 0.3873176571109559
p_occupied_regular (method 2): 0.4388272796659633
p_occupied_intensive (method 2): 0.7533348406059236
p_occupied_inpatient (method 2): 0.38784105475187697
mean_relocated_regular: 977.4
mean_relocated_intensive: 157.4
mean_relocated_inpatient: 847.2
mean_relocated (method 1) 1982.0
mean_relocated (method 2): 1982.0
mean_utilization_A: 0.8438381735196611
mean_utilization_B: 0.8653440068625411
mean_utilization_C: 0.9775903455889423
